<a href="https://colab.research.google.com/github/tranduchoa518-lang/fake-news-detection/blob/main/notebooks/05_welfake_dataset_loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file == 'WELFake_Dataset.csv':
            print(os.path.join(root, file))

/content/drive/MyDrive/WELFake_Dataset.csv


In [3]:
import pandas as pd

file_path = '/content/drive/MyDrive/WELFake_Dataset.csv'

df = pd.read_csv(file_path)

print("Kích thước dữ liệu:", df.shape)
display(df.head())

Kích thước dữ liệu: (72134, 4)


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
print("Số dòng:", len(df))

print("\nSố lượng giá trị bị thiếu:")
print(df.isnull().sum())

print("\nPhân bố nhãn:")
print(df['label'].value_counts())

Số dòng: 72134

Số lượng giá trị bị thiếu:
Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

Phân bố nhãn:
label
1    37106
0    35028
Name: count, dtype: int64


In [5]:

# ==========================================
# 1. LÀM SẠCH DỮ LIỆU WELFAKE
# ==========================================

import pandas as pd

# Tạo bản sao dữ liệu gốc
df_clean = df.copy()

# Chuẩn hóa tên cột
df_clean.columns = df_clean.columns.str.strip().str.lower()

# Kiểm tra các cột cần thiết
print("Các cột trong dữ liệu:", df_clean.columns.tolist())

required_columns = ['title', 'text', 'label']

for col in required_columns:
    if col not in df_clean.columns:
        raise ValueError(f"Không tìm thấy cột {col}. Hãy kiểm tra lại dữ liệu.")

# Kiểm tra nhãn
print("\nSố lượng nhãn ban đầu:")
print(df_clean['label'].value_counts(dropna=False))

# Xử lý dữ liệu thiếu
df_clean['title'] = df_clean['title'].fillna('')
df_clean['text'] = df_clean['text'].fillna('')

# Kết hợp tiêu đề và nội dung bài báo
df_clean['content'] = (
    df_clean['title'].astype(str) + ' ' +
    df_clean['text'].astype(str)
).str.strip()

# Loại bỏ bài báo không có nội dung
df_clean = df_clean[df_clean['content'] != '']

# Loại bỏ dữ liệu trùng lặp
df_clean = df_clean.drop_duplicates(subset=['content'])

# Chỉ giữ lại nhãn 0 và 1
df_clean = df_clean[df_clean['label'].isin([0, 1])]

# Chỉ giữ hai cột cần thiết
df_clean = df_clean[['content', 'label']].reset_index(drop=True)

print("\nKích thước dữ liệu sau khi làm sạch:", df_clean.shape)
print("\nSố lượng nhãn sau khi làm sạch:")
print(df_clean['label'].value_counts())

print("\n5 dòng dữ liệu sau khi làm sạch:")
display(df_clean.head())


Các cột trong dữ liệu: ['unnamed: 0', 'title', 'text', 'label']

Số lượng nhãn ban đầu:
label
1    37106
0    35028
Name: count, dtype: int64

Kích thước dữ liệu sau khi làm sạch: (63676, 2)

Số lượng nhãn sau khi làm sạch:
label
0    34790
1    28886
Name: count, dtype: int64

5 dòng dữ liệu sau khi làm sạch:


,content,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
1,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",0
4,SATAN 2: Russia unvelis an image of its terrif...,1


In [6]:

# ==========================================
# 2. CHIA DỮ LIỆU TRAIN VÀ TEST
# ==========================================

from sklearn.model_selection import train_test_split

# X: Nội dung bài báo
X = df_clean['content']

# y: Nhãn của bài báo
y = df_clean['label']

# Chia dữ liệu:
# 80% dùng để huấn luyện
# 20% dùng để kiểm tra

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Số lượng dữ liệu Train:", len(X_train))
print("Số lượng dữ liệu Test:", len(X_test))

print("\nTỷ lệ Train:", len(X_train) / len(df_clean))
print("Tỷ lệ Test:", len(X_test) / len(df_clean))


Số lượng dữ liệu Train: 50940
Số lượng dữ liệu Test: 12736

Tỷ lệ Train: 0.7999874363967586
Tỷ lệ Test: 0.2000125636032414


In [7]:

# ==========================================
# 3. LƯU DỮ LIỆU ĐÃ XỬ LÝ
# ==========================================

import os

# Tạo thư mục lưu dữ liệu
save_dir = '/content/drive/MyDrive/FakeNewsDetection/processed'

os.makedirs(save_dir, exist_ok=True)

# Lưu toàn bộ dữ liệu đã làm sạch
df_clean.to_csv(
    os.path.join(save_dir, 'welfake_clean.csv'),
    index=False
)

# Tạo bảng Train
train_df = pd.DataFrame({
    'content': X_train,
    'label': y_train
})

# Tạo bảng Test
test_df = pd.DataFrame({
    'content': X_test,
    'label': y_test
})

# Lưu dữ liệu Train
train_df.to_csv(
    os.path.join(save_dir, 'welfake_train.csv'),
    index=False
)

# Lưu dữ liệu Test
test_df.to_csv(
    os.path.join(save_dir, 'welfake_test.csv'),
    index=False
)

print("ĐÃ LƯU THÀNH CÔNG 3 TỆP CSV!")
print("\nThư mục lưu:", save_dir)

print("\nCác tệp đã lưu:")
print(os.listdir(save_dir))


ĐÃ LƯU THÀNH CÔNG 3 TỆP CSV!

Thư mục lưu: /content/drive/MyDrive/FakeNewsDetection/processed

Các tệp đã lưu:
['welfake_clean.csv', 'welfake_train.csv', 'welfake_test.csv']


In [8]:

# ==========================================
# 4. KIỂM TRA CÁC TỆP ĐÃ LƯU
# ==========================================

clean_check = pd.read_csv(
    os.path.join(save_dir, 'welfake_clean.csv')
)

train_check = pd.read_csv(
    os.path.join(save_dir, 'welfake_train.csv')
)

test_check = pd.read_csv(
    os.path.join(save_dir, 'welfake_test.csv')
)

print("Dữ liệu Clean:", clean_check.shape)
print("Dữ liệu Train:", train_check.shape)
print("Dữ liệu Test:", test_check.shape)

print("\nKiểm tra cột:")
print(clean_check.columns.tolist())

print("\nKiểm tra dữ liệu Train:")
display(train_check.head())

print("\nKiểm tra dữ liệu Test:")
display(test_check.head())


Dữ liệu Clean: (63676, 2)
Dữ liệu Train: (50940, 2)
Dữ liệu Test: (12736, 2)

Kiểm tra cột:
['content', 'label']

Kiểm tra dữ liệu Train:


,content,label
0,Internal BLM Memo Leaked: Trump’s Plan to Make...,0
1,OBAMA COZIES UP TO ANOTHER COMMUNIST LEADER TO...,1
2,LIBERAL LUNATIC CHRIS MATTHEWS Scolded By Pier...,1
3,Populism Is a Magnificent Strategy for Liberty...,1
4,Bernie Sanders Has PERFECT Response To Man Ran...,1



Kiểm tra dữ liệu Test:


,content,label
0,EPIC: New Yorker’s “Hillary Themed” Haunted Ho...,1
1,DOZENS Of GOP Foreign Policy Experts Pledge To...,1
2,WATCH HILARIOUS SNL “Draw Muhammed” Contest Sk...,1
3,A Rising Black Leader Who Pulled Off His Own F...,0
4,"G.O.P. Plans Immediate Repeal of Health Law, T...",0


In [9]:
import os
import pandas as pd

from sklearn.model_selection import train_test_split

# Đường dẫn dữ liệu gốc
DATA_PATH = "/content/drive/MyDrive/WELFake_Dataset.csv"

# Đọc dữ liệu
df = pd.read_csv(DATA_PATH)

print("Kích thước dữ liệu:", df.shape)
print("Các cột:", df.columns.tolist())

# Kết hợp title và text nếu có
if "title" in df.columns and "text" in df.columns:
    df["text"] = (
        df["title"].fillna("").astype(str)
        + " "
        + df["text"].fillna("").astype(str)
    )
elif "text" not in df.columns:
    raise ValueError("Không tìm thấy cột text hoặc title.")

# Kiểm tra nhãn
if "label" not in df.columns:
    raise ValueError("Không tìm thấy cột label.")

# Làm sạch dữ liệu
df = df[["text", "label"]].copy()

df["text"] = df["text"].fillna("").astype(str).str.strip()

df = df[df["text"] != ""]
df = df.dropna(subset=["label"])

df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

# Kiểm tra nhãn
print("Phân bố nhãn:")
print(df["label"].value_counts())

print("Dữ liệu sau xử lý:", df.shape)

Kích thước dữ liệu: (72134, 4)
Các cột: ['Unnamed: 0', 'title', 'text', 'label']
Phân bố nhãn:
label
1    37106
0    35028
Name: count, dtype: int64
Dữ liệu sau xử lý: (72134, 2)


In [10]:
# Chia Train và phần còn lại
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

# Chia Validation và Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (50493, 2)
Validation: (10820, 2)
Test: (10821, 2)


In [11]:
OUTPUT_DIR = "/content/drive/MyDrive/fake-news-detection/data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_df.to_csv(
    os.path.join(OUTPUT_DIR, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(OUTPUT_DIR, "val.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(OUTPUT_DIR, "test.csv"),
    index=False
)

print("ĐÃ LƯU THÀNH CÔNG!")

print(os.listdir(OUTPUT_DIR))

ĐÃ LƯU THÀNH CÔNG!
['train.csv', 'val.csv', 'test.csv', 'isot_combined.csv', 'isot_train.csv', 'isot_val.csv', 'isot_test.csv']


In [12]:
# Kiểm tra số dòng trước khi xóa trùng
print("Số dòng trước khi xóa:", len(df))

# Xóa các dòng dữ liệu trùng lặp
df = df.drop_duplicates()

# Đặt lại chỉ mục
df = df.reset_index(drop=True)

# Kiểm tra kết quả
print("Số dòng sau khi xóa:", len(df))
print("Số dòng trùng còn lại:", df.duplicated().sum())

Số dòng trước khi xóa: 72134
Số dòng sau khi xóa: 63676
Số dòng trùng còn lại: 0
